In [1]:
import pandas as pd
from statsmodels.tsa.ardl import ARDL

df = pd.read_csv(
    "../data/processed/passthrough_dataset.csv",
    parse_dates=["date"]
).set_index("date")

# Create the policy dummy again (monthly index)
df["ron95_policy"] = (df.index >= "2025-09-01").astype(int)

# Keep required variables
df = df[["ron95", "wti", "usdmyr", "ron95_policy"]].dropna()

# Ensure monthly frequency (pick ONE; use ME if your pandas warns about M)
df = df.asfreq("ME")   # or df.asfreq("ME") if you're standardizing to month-end

y = df["ron95"]
X = df[["wti", "usdmyr", "ron95_policy"]]

model = ARDL(
    y,
    lags=3,
    exog=X,
    order={"wti": 1, "usdmyr": 1, "ron95_policy": 2},
    trend="c"
)
res = model.fit()
print(res.summary())


                              ARDL Model Results                              
Dep. Variable:                  ron95   No. Observations:                   71
Model:               ARDL(3, 1, 1, 2)   Log Likelihood                 157.659
Method:               Conditional MLE   S.D. of innovations              0.024
Date:                Thu, 08 Jan 2026   AIC                           -291.317
Time:                        09:33:05   BIC                           -264.683
Sample:                    04-30-2020   HQIC                          -280.764
                         - 11-30-2025                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.1994      0.086      2.307      0.025       0.026       0.372
ron95.L1            1.6444      0.063     26.111      0.000       1.518       1.770
ron95.L2           -1.1921      

In [2]:
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.tsatools import lagmat
from statsmodels.stats.diagnostic import het_breuschpagan


In [3]:
def bg_test_ardl_aligned(res, nlags=2):
    X0_full = np.asarray(res.model.data.exog)
    u = np.asarray(res.resid)

    n_u = len(u)
    if n_u <= nlags + 5:
        raise ValueError(f"Too few residual observations ({n_u}) for nlags={nlags}.")

    # Align exog to residual sample
    X0 = X0_full[-n_u:, :]

    # Auxiliary regression: u_t on X + lagged u
    U_lags = lagmat(u, maxlag=nlags, trim="both")
    y = u[nlags:]
    X = np.column_stack([X0[nlags:, :], U_lags])

    aux_res = sm.OLS(y, X).fit()

    n = aux_res.nobs
    lm = n * aux_res.rsquared
    lm_p = stats.chi2.sf(lm, nlags)

    k = X.shape[1]
    R = np.zeros((nlags, k))
    R[:, -nlags:] = np.eye(nlags)
    f_test = aux_res.f_test(R)

    return {
        "LM stat": float(lm),
        "LM p-value": float(lm_p),
        "F stat": float(f_test.fvalue),
        "F p-value": float(f_test.pvalue),
        "nobs_aux": int(n),
        "resid_len": int(n_u),
        "exog_rows_full": int(X0_full.shape[0]),
    }

bg_out = bg_test_ardl_aligned(res, nlags=2)
bg_out


{'LM stat': 1.7114539724665587,
 'LM p-value': 0.42497412841287696,
 'F stat': 0.6315569518077613,
 'F p-value': 0.5352041157173004,
 'nobs_aux': 66,
 'resid_len': 68,
 'exog_rows_full': 71}

In [4]:
def bp_test_ardl(res):
    u = np.asarray(res.resid)
    X0_full = np.asarray(res.model.data.exog)
    X0 = X0_full[-len(u):, :]
    X0 = sm.add_constant(X0, has_constant="add")

    bp = het_breuschpagan(u, X0)

    return {
        "LM stat": float(bp[0]),
        "LM p-value": float(bp[1]),
        "F stat": float(bp[2]),
        "F p-value": float(bp[3]),
        "k_exog": int(X0.shape[1]),
    }

bp_out = bp_test_ardl(res)
bp_out


{'LM stat': 19.03005559581458,
 'LM p-value': 0.0002695144627425167,
 'F stat': 8.29027936046992,
 'F p-value': 9.706152075035667e-05,
 'k_exog': 4}

In [5]:
import joblib
joblib.dump(res, "../data/joblib/ardl_ron95_wti.joblib")

['../data/joblib/ardl_ron95_wti.joblib']